In [164]:
!pip install groq

In [165]:
from google.colab import userdata

In [166]:
from groq import Groq

In [167]:
client = Groq(
    api_key=userdata.get("GROQ_API_KEY"),
)

In [168]:
import json

In [169]:
import requests
from pprint import pprint

In [170]:
def get_weather(location):
  api_key = userdata.get("openweather_API_KEY")
  url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&units=metric&appid={api_key}"

  response = requests.get(url)
  data = response.json()
  if data.get("cod") == 200:
    return json.dumps({
        "location": location,
        "temperature": data["main"]["temp"],
        "description": data["weather"][0]["description"],
    })
  else:
    return json.dumps({
      "oops! something went wrong."
      })

In [171]:
print(get_weather("Jaipur"))

{"location": "Jaipur", "temperature": 32.62, "description": "haze"}


In [172]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current weather for a city",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "City name like Mumbai, London"
          }},
        "required": ["location"]
      }}}
]

In [173]:
llm_messages = [
    {
        "role": "system",
        "content": "You are a helpful weather assistant, use get_weather function when asked about weather"
    },
    {
        "role": "user",
        "content": "What is the weather of Jaipur?"
    }
]

In [174]:
response = client.chat.completions.create(
    messages = llm_messages,
    model = "llama-3.3-70b-versatile",
    tools = tools,
    tool_choice = "auto",
)

In [175]:
response_message = response.choices[0].message
if response_message.tool_calls:
  tool_call = response_message.tool_calls[0]
  arguments = json.loads(tool_call.function.arguments)
  location = arguments['location']
  weather_data = get_weather(location)
  # print(f"weather_data at : {location} : {weather_data}")

  llm_messages.append({
      "role" : "tool",
      "tool_call_id" : tool_call.id,
      "content" : json.dumps(weather_data)
  })

  final_response = client.chat.completions.create(
      messages = llm_messages,
      model = "llama-3.3-70b-versatile",
  )
  print(final_response.choices[0].message.content)

The current weather in Jaipur is 32.62 degrees Celsius with a haze. Would you like to know more about the forecast or any other weather-related information?


In [176]:
  print(response.model_dump_json(indent = 2))

{
  "id": "chatcmpl-dcfe9a94-1e38-448c-a602-984857afe8ea",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "role": "assistant",
        "annotations": null,
        "executed_tools": null,
        "function_call": null,
        "reasoning": null,
        "tool_calls": [
          {
            "id": "d9x9f4khx",
            "function": {
              "arguments": "{\"location\":\"Jaipur\"}",
              "name": "get_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1779398898,
  "model": "llama-3.3-70b-versatile",
  "object": "chat.completion",
  "mcp_list_tools": null,
  "service_tier": "on_demand",
  "system_fingerprint": "fp_3272ea2d91",
  "usage": {
    "completion_tokens": 15,
    "prompt_tokens": 244,
    "total_tokens": 259,
    "completion_time": 0.027745121,
    "completion_tokens_details": null,
    "prompt_time"